In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# 1. PostgreSQL 연결(커넥션 유지)
conn = psycopg2.connect(
    host='host',
    database='postgres',
    user='user',
    password='password',
    port=5432
)

cursor = conn.cursor()

In [2]:
sql_holiday = "SELECT date, date_kind FROM silver.silver_holiday_list;"
df_holiday = pd.read_sql(sql_holiday, conn)

sql_weather_data = "SELECT * FROM silver.api_silver_hourly_weather_forecast;"
df_weather_data = pd.read_sql(sql_weather_data, conn)

sql_filght_data = "SELECT flight_number, scheduled_time, destination, terminal, scheduled_date FROM silver.api_flight_schedule;"
df_filght_data = pd.read_sql(sql_filght_data, conn)

C:\Users\LSS\AppData\Local\Temp\ipykernel_24952\4171065164.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_holiday = pd.read_sql(sql_holiday, conn)
C:\Users\LSS\AppData\Local\Temp\ipykernel_24952\4171065164.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_weather_data = pd.read_sql(sql_weather_data, conn)
C:\Users\LSS\AppData\Local\Temp\ipykernel_24952\4171065164.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_filght_data = pd.read_sql(sql_filght_data, conn)


In [3]:
print(df_holiday)

         date date_kind
0    20230701        01
1    20230702        01
2    20230708        01
3    20230709        01
4    20230715        01
..        ...       ...
252  20250713        01
253  20250719        01
254  20250720        01
255  20250726        01
256  20250727        01

[257 rows x 2 columns]


In [4]:
print(df_weather_data)

          date hour temperature wind_direction wind_speed visibility
0   2025-07-24   18         NaN            330          6       9999
1   2025-07-24   19         NaN            330          6       9999
2   2025-07-24   20         NaN            330          6       9999
3   2025-07-24   21         NaN            330          6       9999
4   2025-07-24   22         NaN            330          6       9999
5   2025-07-24   23         NaN            330          6       9999
6   2025-07-25   00         NaN            330          6       9999
7   2025-07-25   01         NaN            330          6       9999
8   2025-07-25   02         NaN            330          6       9999
9   2025-07-25   03         NaN            330          6       9999
10  2025-07-25   04         NaN            330          6       9999
11  2025-07-25   05         NaN            330          6       9999
12  2025-07-25   06         NaN            330          6       9999
13  2025-07-25   07         NaN   

In [5]:
print(df_filght_data)

     flight_number scheduled_time destination terminal scheduled_date
0           3U7024          16:45         CAN       T1     2025-07-18
1           3U7032          15:15         CAN       T1     2025-07-18
2           3U7034          10:20         CAN       T1     2025-07-18
3            5J129          22:00         CEB       T1     2025-07-18
4            5J187          01:15         MNL       T1     2025-07-18
...            ...            ...         ...      ...            ...
8733        ZH3308          08:30         CAN       T1     2025-07-25
8734        ZH3310          14:50         PEK       T1     2025-07-25
8735         ZH628          14:50         WUX       T1     2025-07-25
8736         ZH632          16:40         SZX       T1     2025-07-25
8737         ZH634          14:40         SZX       T1     2025-07-25

[8738 rows x 5 columns]


In [6]:
# scheduled_date 컬럼명을 'date'로 변경
df_flight_data_renamed = df_filght_data.rename(columns={'scheduled_date': 'date'})

# date 컬럼을 문자열(YYYY-MM-DD)로 통일
df_flight_data_renamed['date'] = pd.to_datetime(df_flight_data_renamed['date']).dt.strftime('%Y-%m-%d')
df_holiday['date'] = pd.to_datetime(df_holiday['date']).dt.strftime('%Y-%m-%d')
df_weather_data['date'] = pd.to_datetime(df_weather_data['date']).dt.strftime('%Y-%m-%d')

# scheduled_time에서 시간만 추출하여 hour 컬럼 생성
df_flight_data_renamed['hour'] = df_flight_data_renamed['scheduled_time'].str[:2]

# date, hour 기준으로 df_weather_data와 inner join
df_joined = pd.merge(
    df_flight_data_renamed,
    df_weather_data,
    how='inner',
    on=['date', 'hour']
)

# df_holiday의 컬럼명 변경 후 date 기준 left join
df_joined = pd.merge(
    df_joined,
    df_holiday,
    how='left',
    on='date'
)

# holiday_kind가 null인 경우 '00'으로 채움
df_joined['date_kind'] = df_joined['date_kind'].fillna('00')

# 결과 확인
print(df_joined)

     flight_number scheduled_time destination terminal        date hour  \
0            5J129          22:00         CEB       T1  2025-07-24   22   
1           7C2103          19:05         MNL       T1  2025-07-24   19   
2           7C2107          21:35         CRK       T1  2025-07-24   21   
3           7C2113          20:50         CEB       T1  2025-07-24   20   
4           7C2125          21:10         TAG       T1  2025-07-24   21   
...            ...            ...         ...      ...         ...  ...   
1095        ZH3308          08:30         CAN       T1  2025-07-25   08   
1096        ZH3310          14:50         PEK       T1  2025-07-25   14   
1097         ZH628          14:50         WUX       T1  2025-07-25   14   
1098         ZH632          16:40         SZX       T1  2025-07-25   16   
1099         ZH634          14:40         SZX       T1  2025-07-25   14   

     temperature wind_direction wind_speed visibility date_kind  
0            NaN            330  

In [7]:
sql_congestion = "SELECT date, hour_of_day, terminal, passengers_forecast FROM gold.api_gold_departure_forecast;"
df_congestion = pd.read_sql(sql_congestion, conn)

C:\Users\LSS\AppData\Local\Temp\ipykernel_24952\4026542275.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_congestion = pd.read_sql(sql_congestion, conn)


In [8]:
print(df_congestion)

         date hour_of_day terminal  passengers_forecast
0    20250720          00       T1                    0
1    20250720          00       T1                  817
2    20250720          00       T1                    0
3    20250720          00       T1                    0
4    20250720          00       T2                    0
..        ...         ...      ...                  ...
427  20250720          23       T1                  941
428  20250720          23       T1                    0
429  20250720          23       T1                    0
430  20250720          23       T2                    1
431  20250720          23       T2                    0

[432 rows x 4 columns]


In [9]:
df_congestion['date'] = pd.to_datetime(df_congestion['date']).dt.strftime('%Y-%m-%d')

df_congestion_grouped = df_congestion.groupby(['date', 'terminal', 'hour_of_day'], as_index=False)['passengers_forecast'].sum()
print(df_congestion_grouped)

          date terminal hour_of_day  passengers_forecast
0   2025-07-20       T1          00                 2451
1   2025-07-20       T1          01                  579
2   2025-07-20       T1          02                   24
3   2025-07-20       T1          03                  630
4   2025-07-20       T1          04                 2718
5   2025-07-20       T1          05                 7614
6   2025-07-20       T1          06                13338
7   2025-07-20       T1          07                16917
8   2025-07-20       T1          08                14184
9   2025-07-20       T1          09                12774
10  2025-07-20       T1          10                12039
11  2025-07-20       T1          11                10095
12  2025-07-20       T1          12                 9519
13  2025-07-20       T1          13                10542
14  2025-07-20       T1          14                 9183
15  2025-07-20       T1          15                 9729
16  2025-07-20       T1        

In [10]:
def get_congestion_level(passengers):
    if passengers <= 2500:
        return '여유'
    elif passengers <= 5000:
        return '보통'
    else:
        return '혼잡'

df_congestion_grouped['congestion_level'] = df_congestion_grouped['passengers_forecast'].apply(get_congestion_level)
print(df_congestion_grouped)

          date terminal hour_of_day  passengers_forecast congestion_level
0   2025-07-20       T1          00                 2451               여유
1   2025-07-20       T1          01                  579               여유
2   2025-07-20       T1          02                   24               여유
3   2025-07-20       T1          03                  630               여유
4   2025-07-20       T1          04                 2718               보통
5   2025-07-20       T1          05                 7614               혼잡
6   2025-07-20       T1          06                13338               혼잡
7   2025-07-20       T1          07                16917               혼잡
8   2025-07-20       T1          08                14184               혼잡
9   2025-07-20       T1          09                12774               혼잡
10  2025-07-20       T1          10                12039               혼잡
11  2025-07-20       T1          11                10095               혼잡
12  2025-07-20       T1          12   

In [11]:
# hour_of_day 컬럼명을 hour로 변경
df_congestion_grouped = df_congestion_grouped.rename(columns={'hour_of_day': 'hour'})

# date, hour, terminal 모두 문자열로 변환 및 strip
df_joined['date'] = df_joined['date'].astype(str).str.strip()
df_joined['hour'] = df_joined['hour'].astype(str).str.zfill(2).str.strip()
df_joined['terminal'] = df_joined['terminal'].astype(str).str.strip()

df_congestion_grouped['date'] = df_congestion_grouped['date'].astype(str).str.strip()
df_congestion_grouped['hour'] = df_congestion_grouped['hour'].astype(str).str.zfill(2).str.strip()
df_congestion_grouped['terminal'] = df_congestion_grouped['terminal'].astype(str).str.strip()

# merge
df_joined = pd.merge(
    df_joined,
    df_congestion_grouped[['date', 'hour', 'terminal', 'congestion_level']],
    on=['date', 'hour', 'terminal'],
    how='left'
)

print(df_joined)

     flight_number scheduled_time destination terminal        date hour  \
0            5J129          22:00         CEB       T1  2025-07-24   22   
1           7C2103          19:05         MNL       T1  2025-07-24   19   
2           7C2107          21:35         CRK       T1  2025-07-24   21   
3           7C2113          20:50         CEB       T1  2025-07-24   20   
4           7C2125          21:10         TAG       T1  2025-07-24   21   
...            ...            ...         ...      ...         ...  ...   
1095        ZH3308          08:30         CAN       T1  2025-07-25   08   
1096        ZH3310          14:50         PEK       T1  2025-07-25   14   
1097         ZH628          14:50         WUX       T1  2025-07-25   14   
1098         ZH632          16:40         SZX       T1  2025-07-25   16   
1099         ZH634          14:40         SZX       T1  2025-07-25   14   

     temperature wind_direction wind_speed visibility date_kind  \
0            NaN            330 

In [12]:
# df_joined에서 cogestion_level이 null인 행 제거
df_joined = df_joined[df_joined['congestion_level'].notnull()]
print(df_joined)

Empty DataFrame
Columns: [flight_number, scheduled_time, destination, terminal, date, hour, temperature, wind_direction, wind_speed, visibility, date_kind, congestion_level]
Index: []


In [13]:
table_name = 'gold.df_flight_prediction'
print(f"'{table_name}' 테이블의 모든 데이터를 삭제합니다...")
cursor.execute(f"DELETE FROM {table_name};")

# 데이터프레임을 튜플의 리스트 형태로 변환
tuples = [tuple(x) for x in df_joined.to_numpy()]

# 삽입할 컬럼들을 문자열로 만듦
# 데이터프레임의 컬럼명으로 cols 문자열을 동적으로 생성
cols = ','.join(list(df_joined.columns))
    
# INSERT 쿼리 템플릿 생성
query = f"INSERT INTO {table_name} ({cols}) VALUES %s"

# execute_values를 사용해 데이터 삽입
execute_values(cursor, query, tuples)
    
# 변경 사항을 데이터베이스에 커밋
conn.commit()
    
print(f"'{table_name}' 테이블에 {len(df_joined)}개의 행이 성공적으로 삽입되었습니다.")

'gold.df_flight_prediction' 테이블의 모든 데이터를 삭제합니다...
'gold.df_flight_prediction' 테이블에 0개의 행이 성공적으로 삽입되었습니다.


In [14]:
cursor.close()
conn.close()